# RSNA Knee — Phase 0 training (Kaggle, internet ON)

Clones the repo, installs deps, trains on the **real** competition data mounted as a
competition input, and writes `code/` + `artifacts/` into `/kaggle/working` — which becomes
this notebook's **Output**, i.e. a Kaggle Dataset you can attach to the offline submission
notebook (`notebooks/infer_notebook.ipynb` in the same repo).

**Kaggle settings for this notebook:** Internet **ON**. GPU optional but recommended.

Edit the constants in the next cell if your fork/branch or the competition dataset slug
differ from the defaults.

In [ ]:
import os

REPO_URL = os.environ.get("KNEE_REPO_URL", "https://github.com/manyagupta13/Knee_detection.git")
REPO_BRANCH = os.environ.get("KNEE_REPO_BRANCH", "claude/phase-0-offline-submission-niozs9")
# Once this branch is merged, switch REPO_BRANCH to "main".

CODE_DIR = "/kaggle/working/code"
ARTIFACTS_DIR = "/kaggle/working/artifacts"

# Point this at the competition's input dataset once attached via Add Input.
# ls /kaggle/input to see what Kaggle actually mounted, and fix this if it differs.
DATA_DIR = os.environ.get("KNEE_DATA_DIR", "/kaggle/input/rsna-2026-knee-mri")

print("repo:", REPO_URL, REPO_BRANCH)
print("data:", DATA_DIR)

In [ ]:
!rm -rf {CODE_DIR}
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {CODE_DIR}
!ls {CODE_DIR}

In [ ]:
# Kaggle images already ship numpy/pandas/torch/pillow/scikit-learn; this only
# fills in what's missing (pydicom, timm) without fighting pinned versions.
!pip install -q pydicom timm

In [ ]:
import sys
sys.path.insert(0, CODE_DIR)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

!ls {DATA_DIR}

## Real column order

`config.TARGET_COLUMNS` in the repo is a **placeholder**. Read the real 12 findings from
the competition's `sample_submission.csv` and overwrite it before training, so the model's
output head order matches what `infer_notebook.ipynb` will later write.

In [ ]:
import config

real_columns = config.target_columns(f"{DATA_DIR}/sample_submission.csv")
config.TARGET_COLUMNS = tuple(real_columns)
config.N_TARGETS = len(config.TARGET_COLUMNS)
print(config.TARGET_COLUMNS)

In [ ]:
from train import train

metrics = train(
    data_dir=DATA_DIR,
    out_dir=ARTIFACTS_DIR,
    epochs=int(os.environ.get("KNEE_EPOCHS", "3")),
    batch_size=int(os.environ.get("KNEE_BATCH_SIZE", "4")),
    n_slices=int(os.environ.get("KNEE_N_SLICES", "16")),
    size=int(os.environ.get("KNEE_SIZE", "224")),
    max_series=1,
    backbone="efficientnet_b0",
    pretrained=True,   # internet is ON in this notebook, so ImageNet weights download fine
    num_workers=2,
)
metrics

In [ ]:
# Keep only the .py source (drop tests/fixtures/.git) so the output dataset is small
# and matches exactly what infer_notebook.ipynb imports from KNEE_CODE_DIR.
import shutil
from pathlib import Path

for junk in [".git", "tests", "fixtures", "notebooks", ".gitignore", "pytest.ini"]:
    p = Path(CODE_DIR) / junk
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

print("kept:", sorted(p.name for p in Path(CODE_DIR).iterdir()))
print("artifacts:", sorted(p.name for p in Path(ARTIFACTS_DIR).iterdir()))
print("\nCommit this notebook (Save Version) — its Output becomes a Kaggle Dataset.")
print("Attach that dataset to infer_notebook.ipynb with internet OFF and run it.")